In [ ]:
import time
import requests
import json
from pathlib import Path
import pandas as pd
from data import load_data

In [ ]:
data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\diagnoses_and_following.csv"
dictionary_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\diagnoses_dictionary.csv"
STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\diagnosess_snomed_state.json")
concept_path = r"D:\Python Projects\Hospital readmission risk\data\concepts"
write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\related_diagnoses.csv"
CACHE: dict[str, str] = {}
"""
any null diagnoses will be 0
"""
sql = """
with sec_codes as (
select 
hu.stay_id,
hu.following_stay_id,
diag.main_diagnosis_code as sec_code
from `hospital-readmission-4.helper_tables.helper_utilization` hu
left join `hospital-readmission-4.data_slim.main_diagnoses` diag
on hu.following_stay_id = diag.stay_id
where hu.readmit_90d = 1
)
select 
diag.stay_id,
diag.main_diagnosis_code as main_code,
sec.following_stay_id as fol_stay_id,
sec.sec_code
from `hospital-readmission-4.data_slim.main_diagnoses` diag
inner join sec_codes sec
on diag.stay_id = sec.stay_id
"""
output_cols = ['is_related']
basic_set = {}

In [ ]:
def load_state():

    global CACHE

    if not STATE_PATH.exists():
        return

    with STATE_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    raw_cache = data.get("cache", {})
    CACHE = {code: path for code, path in raw_cache.items()}

In [ ]:
def read_concept(concept_path):

    path = Path(concept_path)

    if not path.exists():
        return {}
    
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def get_concept(concept_id: str) -> dict:

    if concept_id in CACHE:

        return read_concept(CACHE[concept_id])

In [ ]:
def get_parent_ids(concept: dict) -> list[str]:

    """Extract parent conceptIds from the concept JSON (adjust to actual structure)."""

    parents = []

    for rel in concept.get("relationships", []):
        # 'is a' relationship has conceptId 116680003 in SNOMED CT
        if rel.get("type", {}).get("conceptId") == "116680003" and rel.get("active"):

            target = rel.get("target") or {}

            cid = target.get("conceptId")

            if cid:
                
                parents.append(cid)
            
    return parents

In [ ]:
def find_all_ancestors(concept_id: str, targets, basic_set: set):

    c = get_concept(concept_id, targets)
    ancestors : set[str] = set()
    frontier = {concept_id}
    visited = set()

    for depth in range(10):

        new_frontier = {}

        for cid in frontier:

            if cid in visited:
                continue
            
            c = get_concept(cid, targets)

            if c is {}:
                continue

            parents = get_parent_ids(c)

            for parent in parents:

                if parent in basic_set:
                    continue

                ancestors.update(parent)
                new_frontier.update(parent)
            
        if not new_frontier:
            break
        
        frontier = new_frontier
            
    return ancestors

In [ ]:
def get_relation(concept_id1: str, concept_id2: str, basic_set: set):

    ancestors1 = find_all_ancestors(concept_id1, basic_set)
    ancestors2 = find_all_ancestors(concept_id2, basic_set)

    if ancestors1.intersection(ancestors2):

        return ancestors1.intersection(ancestors2)
    
    return 

In [ ]:
def build_relations(data):

    data.set_index('stay_id', inplace = True)
    
    relations = pd.DataFrame(index = data.index)

    for id in relations.index:

        relations.loc[id, 'is_related'] = get_relation(data.loc[id, 'main_code'], data.loc[id, 'sec_code'], basic_set)

In [ ]:
if __name__ == '__main__':

    data = load_data(data_path, sql)
    load_state()
    